# SafeCrop · Fixed L3: which change to the photo helps? (Google Colab)

A farmer's phone can only take an **L3-quality** photo; there is no clearer L2 photo to ask for. So the app keeps the **same L3 photo** and changes it itself: L3 → L3′ (for example brighter) → L3″ (brighter and zoomed in) and so on. This notebook tests **each change on its own** and in sequence, so you can see exactly which change helps, and for which kind of noise.

**What it does**

1. Makes every version of every L3 photo (the changes are listed below) and shows what they look like.
2. Runs a **quality check** on each photo, which says what is wrong with it (too dark, grainy, blurry, compressed) and how badly. An **automatic** version picks the fix for that problem.
3. Asks **CLIP** (database and zero-shot) about every version: no quota, a few minutes on a GPU.
4. Compares each version with the L3 photo as taken, **per noise type**, with a test of whether the difference could be chance.
5. Runs your **pipelines** (L3 → L3′ → L3″, stopping when CLIP is confident), with brightening and zoom in **separate pipelines**, so you can see which one did the work.
6. Optionally asks **Gemini** about a few versions (uses quota).

| Change | What it does | Meant for |
|---|---|---|
| brighten | multiplies every pixel by the same number until the average brightness is normal | dark photos |
| gamma | a curve that lifts dark tones a lot and bright tones a little | dark photos |
| CLAHE | evens out brightness tile by tile (8 × 8 tiles), with a cap so grain isn't blown up | dark or flat photos |
| contrast | stretches the darkest 1% to black and the brightest 1% to white | flat, washed-out photos |
| saturation | makes colours 50% stronger | faded colours (low light) |
| white balance | removes a colour cast | yellow or blue tint |
| denoise | non-local means: averages each small patch with similar patches elsewhere | grain |
| resample | shrinks the photo to half size and enlarges it back, which averages away grain (re-rasterising) | grain |
| smooth | edge-preserving smoothing: flattens grain into even patches but keeps edges (drawing-like) | grain, compression |
| sharpen | unsharp mask: adds back the difference between the photo and a blurred copy | blur |
| deblock | gentle edge-preserving blur that hides JPEG's 8 × 8 blocks | compression |
| zoom | cuts out the central 70% and enlarges it | small leaf, small spots |
| leaf crop | zooms to where the green leaf actually is | leaf not centred |
| auto | the fix the quality check picks for the photo's problem | any photo |

### Before you run

1. **GPU:** *Runtime → Change runtime type → T4 GPU → Save*.
2. **Images:** `noisy_images.zip` in your Drive, as before.
3. **Gemini (optional):** with the `GEMINI_API_KEY` secret, Gemini is asked about the versions in `GEMINI_VARIANTS`, for 15 photos per disease (105 requests per version; the default 4 versions need 420 requests, one day of the free quota). The L3 photo as taken is not asked again: Step 2's answers are reused. Set `GEMINI_VARIANTS = []` to skip Gemini.
4. **Runtime → Run all.** Results go to `SafeCrop/results/l3_variants`.

In [ ]:
# Install anything missing (Colab already has most of it). Safe to run again.
import subprocess, sys
from importlib.metadata import version, PackageNotFoundError

def ensure(pkg, min_major=None):
    try:
        v = version(pkg)
        if min_major is not None and int(v.split(".")[0]) < min_major:
            raise PackageNotFoundError(pkg)
    except PackageNotFoundError:
        print("installing", pkg, "...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", pkg], check=True)

ensure("google-genai", min_major=1)
ensure("transformers")
ensure("openpyxl")
ensure("scikit-learn")
ensure("scikit-image")
try:
    import cv2
except ImportError:
    ensure("opencv-python-headless")
print("google-genai", version("google-genai"), "| scikit-learn", version("scikit-learn"))

In [ ]:
import os, re, io, csv, json, time, glob, math, random, shutil, zipfile, datetime, subprocess
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, Image as ShowImage
from tqdm.auto import tqdm

def short(e, n=300):
    s = str(e).replace("\n", " ")
    return s if len(s) <= n else s[:n] + " ..."

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
print("pandas", pd.__version__, "| numpy", np.__version__)

## Settings

You can leave these as they are.

In [ ]:
# ---- where the images are ----
SOURCE = "auto"      # "auto": look in Google Drive, ask for an upload if not found | "drive" | "upload"
DATASET_PATH = ""
OUT_SUBDIR = "l3_variants"

# ---- the fixed level ----
TEST_LEVELS = ["L3"]             # the level the farmer's phone gives; e.g. ["L2", "L3"] tests both

# ---- CLIP (the same model and vote as Step 2) ----
CLIP_MODEL = "openai/clip-vit-large-patch14"
CLIP_DATABASE_NEIGHBOURS = 5
STOP_CONFIDENCE = 0.8            # in a pipeline, stop when the winning disease has at least this share of the vote

# ---- the versions of each L3 photo: name -> changes, applied in order ----
VARIANTS = {
    "none": [],                                                    # L3 as the farmer took it
    "brighten": ["brighten"], "zoom": ["zoom"], "brighten+zoom": ["brighten", "zoom"],   # your L3' and L3''
    "gamma": ["gamma"], "clahe": ["clahe"], "contrast": ["contrast"], "saturation": ["saturation"],
    "white_balance": ["white_balance"], "denoise": ["denoise"], "resample": ["resample"], "smooth": ["smooth"],
    "sharpen": ["sharpen"], "deblock": ["deblock"], "leaf_crop": ["leaf_crop"],
    "gamma+denoise": ["gamma", "denoise"], "clahe+denoise": ["clahe", "denoise"], "denoise+sharpen": ["denoise", "sharpen"],
}
# the automatic fix for each problem the quality check finds (from the restoration test on our photos)
ROUTE_RULES = {
    "clean": [], "low_light": ["clahe", "denoise"], "gaussian_noise": ["resample"],
    "jpeg_compression": ["deblock"], "motion_blur": ["sharpen"], "defocus_blur": ["sharpen"],
}
# pipelines: versions tried in order until CLIP is confident (brightening and zoom kept apart)
PIPELINES = {
    "L3 → brighten": ["none", "brighten"],
    "L3 → zoom": ["none", "zoom"],
    "L3 → brighten → brighten+zoom": ["none", "brighten", "brighten+zoom"],
    "L3 → zoom → zoom+brighten": ["none", "zoom", "brighten+zoom"],
    "L3 → auto fix → auto fix+zoom": ["none", "auto (rules)", "auto (rules)+zoom"],
}

# ---- Gemini (optional; uses the free daily quota of 500 requests) ----
GEMINI_VARIANTS = ["brighten", "zoom", "brighten+zoom", "auto (rules)"]   # [] = no new Gemini requests
GEMINI_PHOTOS_PER_DISEASE = 15   # 15 photos x 7 diseases = 105 requests per version and level
GEMINI_MODEL = "gemini-3.5-flash-lite"
GEMINI_REQUESTS_PER_MINUTE = 14
SEED = 299

AUTO = ["auto (rules)", "auto (rules)+zoom"]
for v in GEMINI_VARIANTS:
    if v not in VARIANTS and v not in AUTO:
        raise ValueError(f'GEMINI_VARIANTS: "{v}" is not one of the versions in VARIANTS or {AUTO}')
for name, seq in PIPELINES.items():
    for v in seq:
        if v not in VARIANTS and v not in AUTO:
            raise ValueError(f'Pipeline "{name}" uses "{v}", which is not in VARIANTS')

## 1. Get the images

As in Step 2: `noisy_images.zip` from your Drive (or an upload).

In [ ]:
WORK = "/content/safecrop_inference"
DATA_DIR = os.path.join(WORK, "data")
os.makedirs(DATA_DIR, exist_ok=True)

def mount_drive():
    if os.path.isdir("/content/drive/MyDrive"):
        return True
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Couldn't connect to Google Drive:", short(e, 200))
    return os.path.isdir("/content/drive/MyDrive")

def walk(top, max_depth):
    # os.walk that stops max_depth folders below `top` and skips hidden folders
    top = top.rstrip(os.sep)
    base = top.count(os.sep)
    for root, dirs, files in os.walk(top):
        depth = root.count(os.sep) - base
        dirs[:] = sorted(d for d in dirs if not d.startswith(".")) if depth < max_depth else []
        yield root, files

def is_dataset_folder(root, files):
    return "metadata.csv" in files and os.path.isdir(os.path.join(root, "L0"))

def find_dataset_folder(top):
    for root, files in walk(top, max_depth=6):
        if is_dataset_folder(root, files):
            return root
    return None

def unpack(zip_path, dest):
    with zipfile.ZipFile(zip_path) as z:
        members = [m for m in z.namelist()
                   if not m.startswith("__MACOSX") and not os.path.basename(m.rstrip("/")).startswith("._")]
        z.extractall(dest, members)

def prepare(path):
    # returns the folder that holds metadata.csv and L0..L3, unzipping first if needed
    if os.path.isdir(path):
        return find_dataset_folder(path)
    if not (os.path.isfile(path) and zipfile.is_zipfile(path)):
        return None
    dest = os.path.join(DATA_DIR, re.sub(r"[^\w.-]+", "_", os.path.splitext(os.path.basename(path))[0]))
    if os.path.exists(dest):
        shutil.rmtree(dest)
    print("Unzipping", os.path.basename(path), "...")
    unpack(path, dest)
    for _ in range(3):   # handles a zip inside the zip, e.g. the whole 'safecrop dataset' folder downloaded from Drive
        found = find_dataset_folder(dest)
        if found:
            return found
        inner = [os.path.join(r, f) for r, fs in walk(dest, 6) for f in fs if f.lower().endswith(".zip")]
        inner = [p for p in inner if "noisy" in os.path.basename(p).lower()] or inner
        if not inner:
            break
        for p in inner:
            unpack(p, os.path.splitext(p)[0])
            os.remove(p)
    return None

def find_in_drive():
    found = []
    for root, files in walk("/content/drive/MyDrive", max_depth=4):
        if is_dataset_folder(root, files):
            found.append(root)
        for f in files:
            fl = f.lower()
            if fl.endswith(".zip") and "noisy" in fl and "result" not in fl:
                found.append(os.path.join(root, f))
    return sorted(found, key=os.path.getmtime, reverse=True)

DRIVE_OK, dataset_dir, uploaded = False, None, []

if DATASET_PATH:
    if DATASET_PATH.startswith("/content/drive"):
        DRIVE_OK = mount_drive()
    dataset_dir = prepare(DATASET_PATH)
    if dataset_dir is None:
        print("No images found at DATASET_PATH:", DATASET_PATH)

if dataset_dir is None and SOURCE in ("auto", "drive"):
    DRIVE_OK = mount_drive()
    if DRIVE_OK:
        print("Searching your Google Drive for noisy_images ...")
        candidates = find_in_drive()
        for c in candidates:
            print("  found:", c)
        for c in candidates:
            dataset_dir = prepare(c)
            if dataset_dir:
                print("Using:", c)
                break
        if dataset_dir is None:
            print("No noisy_images.zip found in your Drive. (Files in 'Shared with me' only count if you add a shortcut to My Drive.)")

if dataset_dir is None and SOURCE in ("auto", "upload"):
    from google.colab import files
    print("Click 'Choose files' and pick noisy_images.zip. Without Drive, also add the answer files from "
          "earlier notebooks (gemini_3.5_flash_lite_predictions.csv, ..._help_both_predictions.csv).")
    os.makedirs(os.path.join(WORK, "uploads"), exist_ok=True)
    for name, data in files.upload().items():
        p = os.path.join(WORK, "uploads", name)
        with open(p, "wb") as f:
            f.write(data)
        uploaded.append(p)
    for p in uploaded:
        if p.lower().endswith(".zip"):
            dataset_dir = prepare(p)
            if dataset_dir:
                break

if dataset_dir is None:
    raise RuntimeError("No images found. Put noisy_images.zip in your Google Drive (or set SOURCE = 'upload') and run again.")

# ---- read the list of images ----
meta = pd.read_csv(os.path.join(dataset_dir, "metadata.csv"))
need = {"image", "photo_id", "disease", "level"}
if need - set(meta.columns):
    raise RuntimeError(f"metadata.csv is missing the columns {sorted(need - set(meta.columns))}")
for col in ["disease_name", "group", "origin", "noise"]:
    if col not in meta.columns:
        meta[col] = ""
meta = meta.fillna("")
meta["disease_name"] = [n if n else d for n, d in zip(meta.disease_name, meta.disease)]
meta["level"] = meta.level.astype(str)
meta["path"] = [os.path.join(dataset_dir, p) for p in meta.image]
missing = ~meta.path.map(os.path.exists)
if missing.any():
    print(f"Warning: {missing.sum()} images listed in metadata.csv are missing and will be skipped.")
    meta = meta[~missing]
meta = meta.drop_duplicates("image").sort_values(["disease", "photo_id", "level"]).reset_index(drop=True)

# ---- where results go ----
RESULTS_DIR = "/content/drive/MyDrive/SafeCrop/results" if DRIVE_OK else os.path.join(WORK, "results")
OUT_DIR = os.path.join(RESULTS_DIR, OUT_SUBDIR)       # everything this notebook writes
os.makedirs(OUT_DIR, exist_ok=True)
GEMINI_CACHE = os.path.join(RESULTS_DIR, "gemini_3.5_flash_lite_predictions.csv")   # Step 2: plain Gemini answers
HELP_CACHE = os.path.join(RESULTS_DIR, "gemini_help", "gemini_3.5_flash_lite_help_both_predictions.csv")   # Step 2b

def merge_upload(src, dest):
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    old = pd.read_csv(src, dtype=str, keep_default_na=False)
    if os.path.exists(dest):
        old = pd.concat([pd.read_csv(dest, dtype=str, keep_default_na=False), old])
    old.drop_duplicates(["image", "model"], keep="last").to_csv(dest, index=False)
    print("Using", os.path.basename(src))

for p in uploaded:   # answers from earlier notebooks, when working without Drive
    name = os.path.basename(p).lower()
    if name.endswith(".csv") and "gemini" in name:
        merge_upload(p, HELP_CACHE if "help" in name else GEMINI_CACHE)

print(f"\n{len(meta)} images, {meta.photo_id.nunique()} photos, {meta.disease.nunique()} diseases. Results go to {OUT_DIR}")
display(meta.pivot_table(index=["group", "disease_name"], columns="level", values="image", aggfunc="count", fill_value=0))

## 2. Helpers

Shared code: the 33 options, photo tools (fixes and quality measurements), CLIP and Gemini helpers. You don't need to change anything here.

In [ ]:
# key, crop, name shown to Gemini (after "Crop: "), full name used in CLIP's sentences
LABELS = [dict(key=k, crop=c, gemini=g, clip=n) for k, c, g, n in [
    ("apple_scab",                      "apple",      "scab",                                  "apple scab"),
    ("apple_cedar_rust",                "apple",      "cedar apple rust",                      "cedar apple rust"),
    ("apple_alternaria_leaf_spot",      "apple",      "Alternaria leaf spot",                  "apple Alternaria leaf spot"),
    ("apple_powdery_mildew",            "apple",      "powdery mildew",                        "apple powdery mildew"),
    ("apple_healthy",                   "apple",      "healthy",                               "healthy apple"),
    ("potato_late_blight",              "potato",     "late blight",                           "potato late blight"),
    ("potato_early_blight",             "potato",     "early blight",                          "potato early blight"),
    ("potato_healthy",                  "potato",     "healthy",                               "healthy potato"),
    ("grape_leaf_blight",               "grape",      "leaf blight (Isariopsis leaf spot)",    "grape leaf blight"),
    ("grape_black_rot",                 "grape",      "black rot",                             "grape black rot"),
    ("grape_esca",                      "grape",      "esca (black measles)",                  "grape esca"),
    ("grape_downy_mildew",              "grape",      "downy mildew",                          "grape downy mildew"),
    ("grape_healthy",                   "grape",      "healthy",                               "healthy grape"),
    ("pear_leaf_spot",                  "pear",       "leaf spot",                             "pear leaf spot"),
    ("pear_scab",                       "pear",       "scab",                                  "pear scab"),
    ("pear_fire_blight",                "pear",       "fire blight",                           "pear fire blight"),
    ("pear_rust",                       "pear",       "rust",                                  "pear rust"),
    ("pear_healthy",                    "pear",       "healthy",                               "healthy pear"),
    ("rice_stem_borer",                 "rice",       "stem borer damage (dead heart / white head)", "rice stem borer damage"),
    ("rice_blast",                      "rice",       "blast",                                 "rice blast"),
    ("rice_brown_spot",                 "rice",       "brown spot",                            "rice brown spot"),
    ("rice_bacterial_leaf_blight",      "rice",       "bacterial leaf blight",                 "rice bacterial leaf blight"),
    ("rice_sheath_blight",              "rice",       "sheath blight",                         "rice sheath blight"),
    ("rice_healthy",                    "rice",       "healthy",                               "healthy rice"),
    ("jute_mosaic",                     "jute",       "mosaic",                                "jute mosaic"),
    ("jute_dieback",                    "jute",       "dieback",                               "jute dieback"),
    ("jute_stem_rot",                   "jute",       "stem rot",                              "jute stem rot"),
    ("jute_anthracnose",                "jute",       "anthracnose",                           "jute anthracnose"),
    ("jute_healthy",                    "jute",       "healthy",                               "healthy jute"),
    ("black_gram_yellow_mosaic",        "black gram", "yellow mosaic",                         "black gram yellow mosaic"),
    ("black_gram_cercospora_leaf_spot", "black gram", "Cercospora leaf spot",                  "black gram Cercospora leaf spot"),
    ("black_gram_powdery_mildew",       "black gram", "powdery mildew",                        "black gram powdery mildew"),
    ("black_gram_healthy",              "black gram", "healthy",                               "healthy black gram"),
]]

# one line per option: what it looks like on the plant (only the new Gemini predictor sees these)
DESCRIPTIONS = {
    "apple_scab": "olive-green to dark brown velvety spots on the leaves that turn dark and scabby; leaves may pucker",
    "apple_cedar_rust": "bright yellow-orange round spots on the upper leaf, often with a red rim; small tubes or cups underneath",
    "apple_alternaria_leaf_spot": "many small round brown spots with a darker purple-brown rim that can merge into larger dead patches",
    "apple_powdery_mildew": "white powdery coating on leaves and shoot tips; leaves narrow, curled and crinkled",
    "apple_healthy": "evenly green apple leaves with no spots, coating or discoloured patches",
    "potato_late_blight": "large dark brown to black water-soaked patches, often from the leaf tip or edge, with a pale border and white mould underneath in damp weather",
    "potato_early_blight": "brown spots with rings inside like a target, mostly on older lower leaves, often with yellow around them",
    "potato_healthy": "evenly green potato leaves with no spots, blotches or yellowing",
    "grape_leaf_blight": "irregular dark brown to black angular spots that join into large dry dead areas on the leaf",
    "grape_black_rot": "round tan to brown spots with a dark brown rim and tiny black dots inside",
    "grape_esca": "yellow or red-brown stripes between the veins with green along the veins (tiger stripes), parts drying out",
    "grape_downy_mildew": "yellow oily-looking patches on the upper leaf with white downy growth underneath",
    "grape_healthy": "evenly green grape leaves with no spots, stripes or mould",
    "pear_leaf_spot": "many small round grey to brown spots with a dark rim scattered over the leaf",
    "pear_scab": "olive-black velvety spots on leaves and fruit, like apple scab",
    "pear_fire_blight": "leaves and shoot tips turn black or brown as if burnt and stay attached; shoot tip bends like a hook",
    "pear_rust": "bright orange spots on the upper leaf that later form raised bumps or horn-like growths underneath",
    "pear_healthy": "evenly green pear leaves with no spots, scorching or rust",
    "rice_stem_borer": "pest damage: the central shoot dries and turns brown while outer leaves stay green (dead heart), or the grain head stays upright, empty and white (white head)",
    "rice_blast": "diamond or spindle-shaped spots with grey or white centres and brown edges on the leaves",
    "rice_brown_spot": "many small oval brown spots, often with a grey centre and a yellow halo, scattered over the leaves",
    "rice_bacterial_leaf_blight": "yellow to white wavy stripes from the leaf tip and edges; leaves dry out from the tip down",
    "rice_sheath_blight": "oval grey-green to white patches with brown edges on the leaf sheath near the water, like snake skin",
    "rice_healthy": "green upright rice plants with no spots, stripes, dried shoots or white heads",
    "jute_mosaic": "leaves with a patchy yellow and green mosaic pattern, sometimes crinkled or smaller than normal",
    "jute_dieback": "shoots and branches dry out and die from the tip downward; leaves wilt and turn brown or black",
    "jute_stem_rot": "dark brown to black patches on the stem that spread, so the plant wilts or breaks; leaves may get brown spots",
    "jute_anthracnose": "sunken dark brown spots or cracks on the stem and brown spots on the leaves",
    "jute_healthy": "evenly green jute leaves and stems with no spots, mosaic or drying",
    "black_gram_yellow_mosaic": "bright yellow patches mixed with green across the leaves, spreading until leaves turn mostly yellow",
    "black_gram_cercospora_leaf_spot": "small round brown spots with grey centres and red-brown rims on the leaves",
    "black_gram_powdery_mildew": "white powdery patches on the leaves",
    "black_gram_healthy": "evenly green black gram leaves with no spots, yellowing or powder",
}

# a disease folder that isn't in the list above still works: it gets its own option
known = {l["key"] for l in LABELS}
for d, name in meta[["disease", "disease_name"]].drop_duplicates().itertuples(index=False):
    if d not in known:
        crop = name.split()[0].lower() if name else "plant"
        LABELS.append(dict(key=d, crop=crop, gemini=name, clip=name.lower()))
        DESCRIPTIONS.setdefault(d, "")
        print("Added an option for", d)

LABEL_KEYS = [l["key"] for l in LABELS]
LABEL_CROP = {l["key"]: l["crop"] for l in LABELS}
OPTIONS = [f"{l['crop'][0].upper() + l['crop'][1:]}: {l['gemini']}" for l in LABELS]   # what Gemini sees
OPTION_TO_KEY = dict(zip(OPTIONS, LABEL_KEYS))
KEY_TO_OPTION = dict(zip(LABEL_KEYS, OPTIONS))
TARGETS = sorted(meta.disease.unique())

# ===== SafeCrop photo tools: fixes for noisy leaf photos, and quality measurements =====
# Every function takes and returns a 224 x 224 (or any size) RGB photo as a uint8 numpy array.
import numpy as np
import cv2
from PIL import Image


def to_array(img):
    """PIL image or array -> uint8 RGB array."""
    if isinstance(img, Image.Image):
        img = np.asarray(img.convert("RGB"))
    a = np.asarray(img)
    if a.ndim == 2:
        a = np.stack([a] * 3, axis=2)
    return np.ascontiguousarray(a[..., :3]).astype(np.uint8)


def luminance(a):
    """brightness of every pixel, 0 (black) to 1 (white)"""
    return (a.astype(np.float32) @ np.array([0.299, 0.587, 0.114], np.float32)) / 255.0


def noise_sigma(g):
    """Immerkaer's fast estimate of random noise (same as the loop notebook's), 0 to 1 scale"""
    k = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], np.float32)
    conv = cv2.filter2D(g.astype(np.float32), -1, k, borderType=cv2.BORDER_REFLECT)[1:-1, 1:-1]
    H, W = g.shape
    return float(np.sqrt(np.pi / 2) * np.abs(conv).sum() / (6 * (W - 2) * (H - 2)))


# ---------------------------------------------------------------- fixes
def fix_brighten(a, target=0.45, max_gain=4.0):
    """multiply every pixel by the same number so the average brightness reaches `target` (never darkens)"""
    m = float(luminance(a).mean())
    gain = min(max(target / max(m, 1e-3), 1.0), max_gain)
    return np.clip(a.astype(np.float32) * gain, 0, 255).astype(np.uint8)


def fix_gamma(a, target=0.45):
    """gamma curve: lifts dark tones a lot and bright tones a little, so nothing burns out (never darkens)"""
    m = float(np.clip(luminance(a).mean(), 1e-3, 0.999))
    gamma = float(np.clip(np.log(target) / np.log(m), 0.25, 1.0))
    lut = np.clip(255.0 * (np.arange(256) / 255.0) ** gamma, 0, 255).astype(np.uint8)
    return cv2.LUT(a, lut)


def fix_clahe(a, clip=2.0, tiles=8):
    """CLAHE: evens out brightness tile by tile (8 x 8 tiles), with a cap so grain isn't blown up"""
    lab = cv2.cvtColor(a, cv2.COLOR_RGB2LAB)
    lab[..., 0] = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tiles, tiles)).apply(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)


def fix_contrast(a, low=1, high=99):
    """stretch the darkest 1% to black and the brightest 1% to white"""
    g = luminance(a) * 255
    lo, hi = np.percentile(g, [low, high])
    if hi - lo < 5:
        return a.copy()
    return np.clip((a.astype(np.float32) - lo) * 255.0 / (hi - lo), 0, 255).astype(np.uint8)


def fix_denoise(a):
    """non-local means: averages each small patch with similar patches elsewhere; strength follows the measured grain"""
    s = noise_sigma(luminance(a)) * 255
    h = float(np.clip(0.9 * s, 3, 30))
    return cv2.fastNlMeansDenoisingColored(a, None, h, h, 7, 21)


def fix_sharpen(a, amount=1.0, sigma=2.0):
    """unsharp mask: adds back the difference between the photo and a blurred copy, so edges look crisper"""
    blur = cv2.GaussianBlur(a, (0, 0), sigma)
    return cv2.addWeighted(a, 1 + amount, blur, -amount, 0)


def fix_smooth(a):
    """edge-preserving smoothing (mean shift): flattens grain into even patches but keeps edges,
    a bit like turning the photo into a drawing (the closest thing to 'vectorising' a photo)"""
    return cv2.pyrMeanShiftFiltering(a, sp=8, sr=24)


def fix_deblock(a):
    """gentle edge-preserving blur that hides JPEG's 8 x 8 blocks"""
    return cv2.bilateralFilter(a, 7, 40, 5)


def fix_zoom(a, keep=0.7):
    """cut out the central 70% and enlarge it back to full size"""
    H, W = a.shape[:2]
    h, w = int(round(H * keep)), int(round(W * keep))
    y, x = (H - h) // 2, (W - w) // 2
    return cv2.resize(a[y:y + h, x:x + w], (W, H), interpolation=cv2.INTER_CUBIC)


def fix_resample(a, factor=0.5):
    """shrink then enlarge: each small square of pixels is averaged, which cancels some grain"""
    H, W = a.shape[:2]
    small = cv2.resize(a, (max(8, int(W * factor)), max(8, int(H * factor))), interpolation=cv2.INTER_AREA)
    return cv2.resize(small, (W, H), interpolation=cv2.INTER_CUBIC)


def fix_white_balance(a):
    """grey world: scale red, green and blue so their averages match, removing a colour cast"""
    f = a.astype(np.float32)
    means = f.reshape(-1, 3).mean(0)
    return np.clip(f * (means.mean() / np.maximum(means, 1e-3)), 0, 255).astype(np.uint8)


def fix_saturation(a, factor=1.5):
    """make colours 50% stronger (low light fades them)"""
    hsv = cv2.cvtColor(a, cv2.COLOR_RGB2HSV).astype(np.float32)
    hsv[..., 1] = np.clip(hsv[..., 1] * factor, 0, 255)
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)


def fix_leaf_crop(a, margin=0.08):
    """zoom to where the green leaf is (instead of a fixed centre cut); falls back to a centre zoom
    when no clear leaf area is found"""
    H, W = a.shape[:2]
    small = cv2.GaussianBlur(a, (0, 0), 3).astype(np.float32) / 255.0
    exg = 2 * small[..., 1] - small[..., 0] - small[..., 2]
    mask = exg > max(0.05, float(np.percentile(exg, 60)))
    if not (0.08 < mask.mean() < 0.9):
        return fix_zoom(a)
    ys, xs = np.nonzero(mask)
    y0, y1 = np.percentile(ys, [2, 98]).astype(int)
    x0, x1 = np.percentile(xs, [2, 98]).astype(int)
    side = int(max(y1 - y0, x1 - x0) * (1 + 2 * margin))
    side = int(np.clip(side, 0.4 * min(H, W), min(H, W)))
    cy, cx = (y0 + y1) // 2, (x0 + x1) // 2
    y = int(np.clip(cy - side // 2, 0, H - side))
    x = int(np.clip(cx - side // 2, 0, W - side))
    return cv2.resize(a[y:y + side, x:x + side], (W, H), interpolation=cv2.INTER_CUBIC)


FIXES = {
    "brighten": fix_brighten, "gamma": fix_gamma, "clahe": fix_clahe, "contrast": fix_contrast,
    "denoise": fix_denoise, "sharpen": fix_sharpen, "smooth": fix_smooth, "deblock": fix_deblock,
    "zoom": fix_zoom, "resample": fix_resample, "white_balance": fix_white_balance,
    "saturation": fix_saturation, "leaf_crop": fix_leaf_crop,
}


def apply_chain(a, steps):
    """apply fixes in order, e.g. apply_chain(photo, ["brighten", "zoom"])"""
    a = to_array(a)
    for s in steps:
        a = FIXES[s](a)
    return a


# ---------------------------------------------------------------- quality measurements
FEATURES = ["brightness", "dark_share", "bright_share", "contrast", "dynamic_range", "saturation", "colourfulness",
            "grain", "grain_relative", "colour_grain", "sharpness", "fine_detail", "edges",
            "blur_direction", "blockiness", "leaf_cover"]


def measure(img):
    """16 numbers that describe what is wrong (or right) with a photo"""
    a = to_array(img)
    if max(a.shape[:2]) != 224:
        a = cv2.resize(a, (224, 224), interpolation=cv2.INTER_AREA)
    f = a.astype(np.float32) / 255.0
    g = luminance(a)
    out = {}
    out["brightness"] = float(g.mean())
    out["dark_share"] = float((g < 0.12).mean())
    out["bright_share"] = float((g > 0.95).mean())
    out["contrast"] = float(g.std())
    p2, p98 = np.percentile(g, [2, 98])
    out["dynamic_range"] = float(p98 - p2)
    mx, mn = f.max(2), f.min(2)
    out["saturation"] = float(np.where(mx > 0, (mx - mn) / np.maximum(mx, 1e-6), 0).mean())
    rg, yb = f[..., 0] - f[..., 1], 0.5 * (f[..., 0] + f[..., 1]) - f[..., 2]
    out["colourfulness"] = float(np.hypot(rg.std(), yb.std()) + 0.3 * np.hypot(rg.mean(), yb.mean()))
    s = noise_sigma(g)
    out["grain"] = s
    out["grain_relative"] = float(s / max(g.mean(), 0.02))
    ycc = cv2.cvtColor(a, cv2.COLOR_RGB2YCrCb).astype(np.float32) / 255.0
    out["colour_grain"] = float((noise_sigma(ycc[..., 1]) + noise_sigma(ycc[..., 2])) / 2)
    lap = cv2.Laplacian(g, cv2.CV_32F)
    out["sharpness"] = float(np.log(lap.var() + 1e-8))
    P = np.abs(np.fft.fftshift(np.fft.fft2(g - g.mean()))) ** 2
    H, W = g.shape
    yy, xx = np.mgrid[-H // 2:H - H // 2, -W // 2:W - W // 2]
    r = np.hypot(yy / (H / 2), xx / (W / 2))
    band = (r > 0.02) & (r <= 1)
    out["fine_detail"] = float(P[(r > 0.35) & (r <= 1)].sum() / (P[band].sum() + 1e-12))
    u8 = (g * 255).astype(np.uint8)
    med = float(np.median(u8))
    out["edges"] = float((cv2.Canny(u8, int(max(0, 0.66 * med)), int(min(255, 1.33 * med + 1))) > 0).mean())
    # motion blur smears in one direction: compare spectral energy across 12 directions (middle frequencies)
    ang = np.mod(np.arctan2(yy, xx), np.pi)
    mid = (r > 0.08) & (r < 0.5)
    e = np.array([P[mid & (ang >= k * np.pi / 12) & (ang < (k + 1) * np.pi / 12)].mean() for k in range(12)])
    out["blur_direction"] = float(np.log((e.max() + 1e-12) / (e.min() + 1e-12)))
    dh = np.abs(np.diff(g, axis=1))
    dv = np.abs(np.diff(g, axis=0))
    bh = dh[:, 7::8].mean() / (np.delete(dh, np.s_[7::8], axis=1).mean() + 1e-6)
    bv = dv[7::8, :].mean() / (np.delete(dv, np.s_[7::8], axis=0).mean() + 1e-6)
    out["blockiness"] = float((bh + bv) / 2)
    exg = 2 * f[..., 1] - f[..., 0] - f[..., 2]
    out["leaf_cover"] = float((exg > 0.05).mean())
    return out

In [ ]:
# ---- CLIP helpers: the same model, image preparation and vote as Step 2 ----
CLIP = {}

def clip_start():
    # loads CLIP once; returns False if it can't be loaded
    if CLIP:
        return True
    try:
        import torch
        from transformers import CLIPModel, AutoTokenizer
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Loading {CLIP_MODEL} on " + ("the GPU" if device == "cuda" else
              "the CPU (slower; Runtime > Change runtime type > T4 GPU makes it faster)") + " ...")
        model = CLIPModel.from_pretrained(CLIP_MODEL)
        model = (model.half() if device == "cuda" else model.float()).to(device).eval()
        CLIP.update(torch=torch, model=model, tok=AutoTokenizer.from_pretrained(CLIP_MODEL), device=device,
                    dtype=next(model.parameters()).dtype, size=model.config.vision_config.image_size,
                    MEAN=torch.tensor([0.48145466, 0.4578275, 0.40821073]).view(3, 1, 1),
                    STD=torch.tensor([0.26862954, 0.26130258, 0.27577711]).view(3, 1, 1))
        return True
    except Exception as e:
        print("Couldn't load CLIP:", short(e))
        return False

def clip_stop():
    if CLIP:
        dev, torch = CLIP["device"], CLIP["torch"]
        CLIP.clear()
        if dev == "cuda":
            torch.cuda.empty_cache()

def clip_prep(a):
    # exactly Step 2's preparation: shortest side to 224 (bicubic), centre square, CLIP's colour scaling
    torch, size = CLIP["torch"], CLIP["size"]
    img = Image.fromarray(a)
    w, h = img.size
    s = size / min(w, h)
    if s != 1:
        img = img.resize((max(size, round(w * s)), max(size, round(h * s))), Image.BICUBIC)
    w, h = img.size
    left, top = (w - size) // 2, (h - size) // 2
    img = img.crop((left, top, left + size, top + size))
    x = torch.from_numpy(np.asarray(img).astype(np.float32) / 255.0).permute(2, 0, 1)
    return (x - CLIP["MEAN"]) / CLIP["STD"]

def clip_image_vectors(items, batch=32, desc="CLIP: images"):
    # items: photo file paths or RGB arrays; returns one unit-length vector per item
    torch, model = CLIP["torch"], CLIP["model"]
    out = []
    with torch.no_grad():
        for i in tqdm(range(0, len(items), batch), desc=desc):
            chunk = [x if isinstance(x, np.ndarray) else to_array(Image.open(x)) for x in items[i:i + batch]]
            x = torch.stack([clip_prep(a) for a in chunk]).to(device=CLIP["device"], dtype=CLIP["dtype"])
            v = model.visual_projection(model.vision_model(pixel_values=x).pooler_output)
            out.append(torch.nn.functional.normalize(v.float(), dim=-1).cpu())
    return torch.cat(out).numpy()

def clip_text_vectors(texts):
    torch, model = CLIP["torch"], CLIP["model"]
    with torch.no_grad():
        t = CLIP["tok"](texts, padding=True, truncation=True, max_length=77, return_tensors="pt")
        v = model.text_projection(model.text_model(input_ids=t["input_ids"].to(CLIP["device"]),
                                                   attention_mask=t["attention_mask"].to(CLIP["device"])).pooler_output)
        return torch.nn.functional.normalize(v.float(), dim=-1).cpu().numpy()

def option_sentences(l):
    # the same four sentences per option as Step 2's zero-shot
    if l["key"].endswith("_healthy"):
        c = l["crop"]
        return [f"a photo of a healthy {c} plant.", f"a photo of healthy {c} leaves.",
                f"a close-up photo of a healthy {c} leaf.", f"a photo of a {c} plant with no disease."]
    n, c = l["clip"], l["crop"]
    return [f"a photo of {n}.", f"a photo of a plant with {n}.",
            f"a close-up photo of a leaf with {n}.", f"a photo of {c} plants affected by {n}."]

def option_vectors():
    T = np.stack([clip_text_vectors(option_sentences(l)).mean(axis=0) for l in LABELS])
    return T / np.linalg.norm(T, axis=1, keepdims=True)

def database_vote(S, ref_labels, k):
    # S: similarity of each query (rows) to each reference (columns); -inf = not allowed.
    # The k most similar references vote, weighted by similarity (Step 2's rule).
    preds, confs = [], []
    k = max(1, min(k, S.shape[1]))
    nearest = np.argsort(-S, axis=1)[:, :k]
    for i in range(S.shape[0]):
        votes = {}
        for j in nearest[i]:
            if np.isfinite(S[i, j]):
                votes[ref_labels[j]] = votes.get(ref_labels[j], 0.0) + max(float(S[i, j]), 0.0) + 1e-6
        if not votes:
            preds.append(""); confs.append(0.0); continue
        best = max(votes, key=votes.get)
        preds.append(best)
        confs.append(votes[best] / sum(votes.values()))
    return preds, confs, nearest

In [ ]:
# ---- Gemini helpers: the same prompts, pacing, retries and saving as Steps 2 and 2b ----
class DailyLimit(Exception): pass
class KeyProblem(Exception): pass
class EmptyReply(Exception): pass
class NotInFreeTier(Exception): pass

GEM = {}   # filled by gemini_start(): client, types, errors
COLUMNS = ["image", "model", "job", "answer", "pred_label", "confidence", "clear_enough", "references", "seconds", "time"]

def get_api_key():
    for name in ("GEMINI_API_KEY", "GOOGLE_API_KEY"):
        try:
            from google.colab import userdata
            k = userdata.get(name)
            if k:
                return k.strip()
        except Exception:
            pass
        if os.environ.get(name):
            return os.environ[name].strip()
    from getpass import getpass
    print("No Colab secret called GEMINI_API_KEY was found (see the steps at the top).")
    return getpass("Paste your Gemini API key here and press Enter (leave empty to skip Gemini): ").strip()

def gemini_start():
    if GEM:
        return True
    key = get_api_key()
    if not key:
        print("No API key, so Gemini is skipped; the tables use the answers saved so far.")
        return False
    from google import genai
    from google.genai import types, errors
    GEM.update(client=genai.Client(api_key=key), types=types, errors=errors)
    return True

def make_config(options):
    types = GEM["types"]
    return types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=types.Schema(
            type=types.Type.OBJECT,
            properties={"diagnosis": types.Schema(type=types.Type.STRING, enum=list(options)),
                        "confidence": types.Schema(type=types.Type.NUMBER),
                        "clear_enough": types.Schema(type=types.Type.BOOLEAN)},
            required=["diagnosis", "confidence", "clear_enough"]))

def norm_text(s):
    return re.sub(r"[^a-z0-9]+", " ", str(s).lower()).strip()

def key_for(answer, options):
    # the label key of Gemini's answer, matched only against this job's options
    if answer in options:
        return OPTION_TO_KEY[answer]
    norm = {norm_text(o): o for o in options}
    n = norm_text(answer)
    if n in norm:
        return OPTION_TO_KEY[norm[n]]
    hits = [o for k, o in norm.items() if k and k in n]
    return OPTION_TO_KEY[max(hits, key=len)] if hits else ""

def parse_reply(text, options):
    text = (text or "").strip()
    data = {}
    m = re.search(r"\{.*\}", text, re.S)
    if m:
        try:
            data = json.loads(m.group(0))
        except Exception:
            data = {}
    if not isinstance(data, dict):
        data = {}
    answer = data.get("diagnosis") or text
    try:
        conf = float(data.get("confidence"))
        conf = conf / 100 if 1 < conf <= 100 else conf
        conf = min(max(conf, 0.0), 1.0)
    except Exception:
        conf = ""
    clear = data.get("clear_enough", "")
    if isinstance(clear, str):
        clear = {"true": True, "yes": True, "false": False, "no": False}.get(clear.strip().lower(), "")
    return str(answer), key_for(answer, options), conf, clear

def append_row(path, row, columns):
    new = not os.path.exists(path)
    with open(path, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=columns)
        if new:
            w.writeheader()
        w.writerow({c: row.get(c, "") for c in columns})

def load_answers(path, model=None):
    # saved answers in one file, one row per image (the latest)
    empty = pd.DataFrame(columns=["image", "model", "answer", "pred_label", "confidence", "clear_enough"])
    if not os.path.exists(path):
        return empty
    c = pd.read_csv(path, dtype=str, keep_default_na=False)
    if not {"image", "model", "answer", "pred_label"} <= set(c.columns):
        print("Ignoring", os.path.basename(path), "(unexpected columns)")
        return empty
    c = c[(c.model == (model or GEMINI_MODEL)) & ((c.pred_label != "") | (c.answer != ""))]
    return c.drop_duplicates("image", keep="last")

MIME = {".png": "image/png", ".jpg": "image/jpeg", ".jpeg": "image/jpeg", ".webp": "image/webp"}

def part_from_path(path):
    with open(path, "rb") as f:
        data = f.read()
    return GEM["types"].Part.from_bytes(data=data, mime_type=MIME.get(os.path.splitext(path)[1].lower(), "image/png"))

def part_from_array(a, fmt="PNG", max_side=None):
    img = Image.fromarray(a)
    if max_side:
        img.thumbnail((max_side, max_side))
    buf = io.BytesIO()
    img.save(buf, format=fmt, **({"quality": 90} if fmt == "JPEG" else {}))
    return GEM["types"].Part.from_bytes(data=buf.getvalue(), mime_type="image/jpeg" if fmt == "JPEG" else "image/png")

def plain_prompt(options):
    # Step 2's prompt, with the given list of options
    return ("You are an expert plant pathologist. The photo shows part of a crop plant. "
            "It may be blurry, dark, grainy or compressed.\n"
            "Which ONE option below best describes the main disease, pest damage or condition you can see? "
            "Pick the single most likely option, even if you are unsure.\n\nOptions:\n"
            + "\n".join("- " + o for o in options)
            + "\n\nAnswer in JSON with \"diagnosis\" (exactly one option from the list), \"confidence\" "
              "(from 0 to 1) and \"clear_enough\" (true if the photo is clear enough for a reliable diagnosis, "
              "false if the farmer should take it again).")

def help_head(n_refs):
    # Step 2b's opening text
    head = ("You are an expert plant pathologist. The first image is a farmer's photo of part of a crop plant. "
            "It may be blurry, dark, grainy or compressed.\n")
    if n_refs:
        head += (f"After it come {n_refs} reference photos from a labelled collection of clean photos, each with "
                 "its correct label. A similarity search picked them as the photos that look most like the farmer's "
                 "photo, most similar first. Use them as examples to compare with: they can show a different "
                 "disease from the farmer's photo.\n")
    return head

def help_question(options, with_descriptions=True):
    # Step 2b's question, with descriptions next to the options
    lines = "\n".join(f"- {o} — {DESCRIPTIONS[OPTION_TO_KEY[o]]}" if with_descriptions and DESCRIPTIONS.get(OPTION_TO_KEY[o])
                      else f"- {o}" for o in options)
    return ("Which ONE option below best describes the main disease, pest damage or condition you can see in the "
            "farmer's photo? Pick the single most likely option, even if you are unsure.\n\nOptions"
            + (", each with what it looks like" if with_descriptions else "") + ":\n" + lines
            + "\n\nAnswer in JSON with \"diagnosis\" (exactly one option from the list"
            + (", without its description" if with_descriptions else "") + "), \"confidence\" (from 0 to 1) and "
              "\"clear_enough\" (true if the farmer's photo is clear enough for a reliable diagnosis, false if the farmer "
              "should take it again).")

_last_call = [0.0]
_rpm = [float(GEMINI_REQUESTS_PER_MINUTE)]
_busy_in_a_row, _ok_streak = [0], [0]

def pace():
    gap = 60.0 / max(_rpm[0], 0.1)
    wait = _last_call[0] + gap - time.time()
    if wait > 0:
        time.sleep(wait)
    _last_call[0] = time.time()

def ask(contents, config):
    resp = GEM["client"].models.generate_content(model=GEMINI_MODEL, contents=contents, config=config)
    try:
        text = resp.text
    except Exception:
        text = None
    if not text:
        why = ""
        try:
            why = str(resp.candidates[0].finish_reason)
        except Exception:
            pass
        raise EmptyReply(why.strip() or "empty reply")
    return text

def retry_seconds(msg):
    m = re.search(r"retryDelay\W+(\d+(?:\.\d+)?)s", msg) or re.search(r"retry in (\d+(?:\.\d+)?)\s*s", msg, re.I)
    return float(m.group(1)) if m else None

def call_gemini(contents, config, tries=8):
    last = None
    for attempt in range(tries):
        pace()
        try:
            text = ask(contents, config)
            _busy_in_a_row[0] = 0
            _ok_streak[0] += 1
            if _ok_streak[0] % 20 == 0:
                _rpm[0] = min(float(GEMINI_REQUESTS_PER_MINUTE), _rpm[0] * 1.15)
            return text
        except GEM["errors"].APIError as e:
            code, msg, last = getattr(e, "code", None), str(e), e
            if code == 429:
                if re.search(r"limit:\s*0\b", msg):
                    raise NotInFreeTier(msg)
                if re.search(r"PerDay|per day|daily", msg, re.I):
                    raise DailyLimit(msg)
                _busy_in_a_row[0] += 1
                _ok_streak[0] = 0
                if _busy_in_a_row[0] >= 8:
                    raise DailyLimit(msg)
                if _rpm[0] > 2:
                    _rpm[0] = max(2.0, _rpm[0] * 0.8)
                wait = (retry_seconds(msg) or 20 * (attempt + 1)) + 2
                print(f"  too many requests; waiting {wait:.0f} s, then continuing at {_rpm[0]:.1f} per minute ...")
                time.sleep(wait)
                continue
            if code is None or code >= 500:
                wait = min(120, 5 * 2 ** attempt)
                print(f"  Gemini is busy ({code}); trying again in {wait} s ...")
                time.sleep(wait)
                continue
            if code in (401, 403) or re.search(r"API.?key", msg, re.I):
                raise KeyProblem(msg)
            raise
        except EmptyReply as e:
            last = e
            if attempt >= 1:
                raise
        except Exception as e:
            last = e
            wait = min(120, 5 * 2 ** attempt)
            print(f"  connection problem ({short(e, 80)}); trying again in {wait} s ...")
            time.sleep(wait)
    raise RuntimeError(f"gave up after {tries} tries: {short(last, 200)}")

def run_job(name, cache_path, queue, build, options, extra=None):
    """Asks Gemini about every row of `queue` without a saved answer in cache_path.
    build(row) gives the request contents; extra(row) gives extra columns to save.
    Returns 'done', 'daily', 'key' or 'stopped'."""
    done = set(load_answers(cache_path).image)
    todo = [r for r in queue if r.image not in done]
    if not todo:
        print(f"{name}: every image is answered already ({len(queue)}).")
        return "done"
    if not gemini_start():
        return "key"
    config = make_config(options)
    print(f"\n{name}: {len(todo)} images to go, about {len(todo) / _rpm[0]:.0f} minutes at {_rpm[0]:.0f} per minute. "
          "You can stop at any time; progress is saved.")
    n_ok, n_err, errors_in_a_row, status = 0, 0, 0, "done"
    bar = tqdm(total=len(todo), desc=name)
    try:
        for row in todo:
            t0 = time.time()
            try:
                text = call_gemini(build(row), config)
                answer, key, conf, clear = parse_reply(text, options)
                rec = {"image": row.image, "model": GEMINI_MODEL, "job": name, "answer": answer[:300], "pred_label": key,
                       "confidence": conf, "clear_enough": clear, "seconds": round(time.time() - t0, 1),
                       "time": datetime.datetime.now().isoformat(timespec="seconds")}
                if extra:
                    rec.update(extra(row))
                append_row(cache_path, rec, COLUMNS)
                n_ok += 1
                errors_in_a_row = 0
            except DailyLimit:
                print("\nGemini's free quota for today seems to be used up. Progress is saved: run this notebook again "
                      "after midnight Pacific time (about 2 pm in Bangladesh) and it continues where it stopped.")
                status = "daily"
                break
            except (KeyProblem, NotInFreeTier) as e:
                print("\nGemini refused the key or the model:", short(e, 200))
                status = "key"
                break
            except Exception as e:
                n_err += 1
                errors_in_a_row += 1
                append_row(os.path.join(OUT_DIR, "gemini_errors.csv"),
                           {"image": row.image, "model": GEMINI_MODEL, "job": name, "error": short(e, 500),
                            "time": datetime.datetime.now().isoformat(timespec="seconds")},
                           ["image", "model", "job", "error", "time"])
                if (n_ok == 0 and errors_in_a_row >= 2) or errors_in_a_row >= 10:
                    print("\nGemini keeps failing, so it stops for now. Last error:", short(e, 300))
                    status = "stopped"
                    break
            bar.update(1)
    except KeyboardInterrupt:
        print("\nStopped. Answers so far are saved; run this cell again to continue.")
        status = "stopped"
    bar.close()
    total = len(set(load_answers(cache_path).image) & {r.image for r in queue})
    print(f"{name}: {n_ok} answers this run, {n_err} errors. {total} of {len(queue)} images answered in total.")
    return status

def gemini_queue(levels, photos_per_disease=None, skip_photos=()):
    # a fixed random order of photos (balanced partial runs), each with the chosen levels
    photos = meta.drop_duplicates("photo_id")[["photo_id", "disease"]]
    rng = random.Random(SEED)
    keep = []
    for d, ids in photos.groupby("disease").photo_id:
        ids = sorted(i for i in ids if i not in set(skip_photos))
        rng.shuffle(ids)
        keep += ids if photos_per_disease is None else ids[:photos_per_disease]
    keep = sorted(keep)
    random.Random(SEED + 1).shuffle(keep)
    by_photo = {pid: g for pid, g in meta.groupby("photo_id")}
    return [r for pid in keep for r in by_photo[pid].sort_values("level").itertuples(index=False) if r.level in levels]

## 3. What each change does

One apple scab photo per noise type, at the first test level, with each change. Look before trusting the numbers: a change can make a photo look nicer to us without helping the model, and the other way round.

In [ ]:
FIG = os.path.join(OUT_DIR, "figures")
os.makedirs(FIG, exist_ok=True)
NOISE_NAME = {"gaussian_noise": "Grain (sensor noise)", "low_light": "Low light", "jpeg_compression": "Compression",
              "motion_blur": "Hand-shake blur", "defocus_blur": "Out of focus"}
LEVEL_SHOW = TEST_LEVELS[-1]
show_cols = [v for v in ["none", "brighten", "gamma", "clahe", "contrast", "denoise", "resample", "sharpen", "smooth",
                         "zoom", "leaf_crop"] if v in VARIANTS]
example_disease = "apple_scab" if "apple_scab" in set(meta.disease) else meta.disease.iat[0]
fig, axes = plt.subplots(len(NOISE_NAME), len(show_cols) + 1, figsize=(1.5 * (len(show_cols) + 1), 1.65 * len(NOISE_NAME) + 0.5), dpi=130)
for i, nz in enumerate(NOISE_NAME):
    sel = meta[(meta.disease == example_disease) & (meta.noise == nz) & (meta.level == LEVEL_SHOW)]
    if not len(sel):
        continue
    r = sel.iloc[0]
    a = to_array(Image.open(r.path))
    clean = to_array(Image.open(meta[(meta.photo_id == r.photo_id) & (meta.level == "L0")].path.iat[0]))
    for j, name in enumerate(["clean L0"] + show_cols):
        ax = axes[i, j]
        ax.imshow(clean if name == "clean L0" else apply_chain(a, VARIANTS[name]))
        ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values():
            s.set_visible(False)
        if i == 0:
            ax.set_title(name if name != "none" else f"{LEVEL_SHOW} as taken", fontsize=8)
        if j == 0:
            ax.set_ylabel(NOISE_NAME[nz], fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(FIG, "what_each_change_does.png"), facecolor="white")
plt.close(fig)
display(ShowImage(filename=os.path.join(FIG, "what_each_change_does.png"), width=1000))

## 4. The quality check: what is wrong with each photo?

Every photo gets 16 measurements: brightness, how much is very dark, contrast, colour, grain (overall and in the colours), sharpness, fine detail, edges, smearing in one direction, JPEG blockiness and how much of the photo is leaf. A random forest learns from our 840 images which measurements go with which problem (clean, grain, low light, compression, hand-shake blur, out of focus) and how bad it is (L0 to L3).

It is tested fairly: the photos are split into 5 groups, and each group is checked by a forest that never saw any version of those photos. The **automatic fix** for each test photo uses this fair answer, so the automatic pipeline is never helped by a check that already knew the photo.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, cross_val_predict
F = pd.DataFrame([measure(Image.open(p)) for p in tqdm(meta.path, desc="measuring photos")])
meta["problem"] = np.where(meta.level == "L0", "clean", meta.noise)
forest = RandomForestClassifier(n_estimators=500, random_state=0, n_jobs=-1)
meta["detected"] = cross_val_predict(forest, F[FEATURES], meta.problem, cv=GroupKFold(n_splits=5), groups=meta.photo_id)
meta["detected_level"] = cross_val_predict(forest, F[FEATURES], meta.level, cv=GroupKFold(n_splits=5), groups=meta.photo_id)
print(f"Quality check, tested on photos it never saw: problem right for {(meta.detected == meta.problem).mean() * 100:.1f}% "
      f"of images, level right for {(meta.detected_level == meta.level).mean() * 100:.1f}%.")
SHORT = {"clean": "clean", "gaussian_noise": "grain", "low_light": "low light", "jpeg_compression": "compression",
         "motion_blur": "hand-shake blur", "defocus_blur": "out of focus"}
print("\nRows: the real problem; columns: what the check found.")
display(pd.crosstab(meta.problem.map(SHORT), meta.detected.map(SHORT)))
F.insert(0, "image", meta.image)
F.to_csv(os.path.join(OUT_DIR, "quality_measurements.csv"), index=False)
DETECTED = dict(zip(meta.image, meta.detected))
Q = meta[meta.level.isin(TEST_LEVELS)].reset_index(drop=True)      # the test photos
AUTO_STEPS = {img: ROUTE_RULES.get(DETECTED[img], []) for img in Q.image}
print(f"\nAutomatic fix chosen for the {len(Q)} test images:")
display(pd.Series([" + ".join(AUTO_STEPS[i]) or "no change" for i in Q.image]).value_counts().rename("images").to_frame())

## 5. CLIP on every version

Every version of every test photo goes through CLIP. The **database** is the same as in Step 2: our clean (L0) photos, where a photo never finds its own clean version. **Zero-shot** CLIP chooses from the 33 options.

Two automatic versions are included:
* **auto (rules):** the fix from `ROUTE_RULES` for the problem the quality check found.
* **auto (learned):** for each problem, the version that worked best for CLIP **on the other photos** (the same 5 groups as the quality check), so the choice is never made using the photo being tested.

The vectors are saved, so a second run doesn't need CLIP again.

In [ ]:
def slug(s):
    return re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")

def steps_for(variant, image):
    if variant == "auto (rules)":
        return AUTO_STEPS[image]
    if variant == "auto (rules)+zoom":
        return AUTO_STEPS[image] + ["zoom"]
    return VARIANTS[variant]

def variant_array(variant, row):
    return apply_chain(to_array(Image.open(row.path)), steps_for(variant, row.image))

ALL_VARIANTS = list(VARIANTS) + AUTO
CACHE = os.path.join(OUT_DIR, "clip_variants.npz")
E = {}
if os.path.exists(CACHE):
    try:
        z = np.load(CACHE, allow_pickle=False)
        if str(z["model"]) == CLIP_MODEL and [str(x) for x in z["images"]] == list(Q.image) \
                and [str(x) for x in z["route"]] == [json.dumps(ROUTE_RULES, sort_keys=True)]:
            saved = {str(k): str(v) for k, v in zip(z["names"], z["steps"])}
            for v in ALL_VARIANTS + ["reference", "options"]:
                key = slug(v)
                want = json.dumps(VARIANTS.get(v, v))
                if key in z.files and saved.get(v) == want:
                    E[v] = z[key]
    except Exception as e:
        print("The saved CLIP vectors couldn't be used (" + short(e, 80) + ").")
todo = [v for v in ALL_VARIANTS + ["reference", "options"] if v not in E]
if todo:
    if not clip_start():
        raise RuntimeError("CLIP couldn't be loaded. Check the internet connection and run again.")
    if "reference" in todo:
        E["reference"] = clip_image_vectors(list(meta.path[meta.level == "L0"]), desc="CLIP: clean database photos")
    if "options" in todo:
        E["options"] = option_vectors()
    for v in [v for v in todo if v not in ("reference", "options")]:
        E[v] = clip_image_vectors([variant_array(v, r) for r in Q.itertuples(index=False)], desc=f"CLIP: {v}")
    np.savez(CACHE, model=np.array(CLIP_MODEL), images=np.array(list(Q.image)),
             route=np.array([json.dumps(ROUTE_RULES, sort_keys=True)]),
             names=np.array(list(E)), steps=np.array([json.dumps(VARIANTS.get(v, v)) for v in E]),
             **{slug(v): E[v] for v in E})
    clip_stop()
else:
    print("CLIP vectors for every version are already saved, so CLIP isn't loaded again.")

REF = meta[meta.level == "L0"]
REF_PID, REF_DIS = REF.photo_id.to_numpy(), REF.disease.to_numpy()
rows = []
for v in ALL_VARIANTS:
    S = E[v] @ E["reference"].T
    S[Q.photo_id.to_numpy()[:, None] == REF_PID[None, :]] = -np.inf      # never the photo's own clean version
    db_pred, db_conf, _ = database_vote(S, REF_DIS, CLIP_DATABASE_NEIGHBOURS)
    logits = E[v] @ E["options"].T
    zs_pred = [LABEL_KEYS[i] for i in logits.argmax(1)]
    for i, r in enumerate(Q.itertuples(index=False)):
        rows.append(dict(image=r.image, variant=v, model="CLIP database", pred=db_pred[i], conf=db_conf[i]))
        rows.append(dict(image=r.image, variant=v, model="CLIP zero-shot", pred=zs_pred[i], conf=np.nan))
RES = pd.DataFrame(rows).merge(Q[["image", "photo_id", "disease", "group", "noise", "level", "detected"]], on="image")
RES["correct"] = RES.pred == RES.disease

# ---- auto (learned): per problem, the version that worked best on the OTHER photos ----
CANDIDATES = [v for v in VARIANTS]
chosen = {}
for model in ["CLIP database", "CLIP zero-shot"]:
    R = RES[(RES.model == model) & RES.variant.isin(CANDIDATES)]
    acc_img = R.pivot_table(index="image", columns="variant", values="correct", aggfunc="first")[CANDIDATES].astype(float)
    imgs = Q.image.to_numpy()
    for train, test in GroupKFold(n_splits=5).split(Q, groups=Q.photo_id):
        tr = Q.iloc[train]
        for problem, g in tr.groupby("detected"):
            means = acc_img.loc[g.image].mean()
            best = means.idxmax() if means.max() > means.get("none", -1) else "none"
            for img in Q.iloc[test][Q.iloc[test].detected == problem].image:
                chosen[(model, img)] = best
        for img in Q.iloc[test].image:
            chosen.setdefault((model, img), "none")
    pick = R.set_index(["image", "variant"])
    extra_rows = []
    for img in imgs:
        v = chosen[(model, img)]
        r = pick.loc[(img, v)]
        extra_rows.append(dict(image=img, variant="auto (learned)", model=model, pred=r.pred, conf=r.conf, chosen=v))
    RES = pd.concat([RES, pd.DataFrame(extra_rows).merge(
        Q[["image", "photo_id", "disease", "group", "noise", "level", "detected"]], on="image")], ignore_index=True)
RES["correct"] = RES.pred == RES.disease
RES.to_csv(os.path.join(OUT_DIR, "clip_predictions_by_version.csv"), index=False)
learned = RES[RES.variant == "auto (learned)"]
print("auto (learned): the version CLIP database used, by the problem the quality check found")
display(pd.crosstab(learned[learned.model == "CLIP database"].detected.map(SHORT),
                    learned[learned.model == "CLIP database"].chosen))

## 6. Which change helped?

**Table A** gives each version's accuracy for each noise type. **Table B** compares each version with the L3 photo as taken, on the same photos: *fixed* = photos only the changed version got right, *broken* = photos only the original got right, and *p* below 0.05 means the difference is very unlikely to be chance. **Table C** separates brightening from zoom: the effect of brightening is the average over zoom on and off, and the other way round.

In [ ]:
from scipy.stats import binomtest
ORDER = ALL_VARIANTS + ["auto (learned)"]
NOISES = [n for n in NOISE_NAME if n in set(Q.noise)]

def acc_table(model, df=None):
    R = (RES if df is None else df)
    R = R[R.model == model]
    t = R.pivot_table(index="variant", columns="noise", values="correct", aggfunc="mean").mul(100).reindex(columns=NOISES)
    t["All"] = R.groupby("variant").correct.mean().mul(100)
    t = t.reindex([v for v in ORDER if v in set(R.variant)])
    t.columns = [NOISE_NAME.get(c, c) for c in t.columns]
    return t

def versus_none(model, df=None):
    R = (RES if df is None else df)
    W = R[R.model == model].pivot_table(index="image", columns="variant", values="correct", aggfunc="first").astype(bool)
    out = []
    if "none" not in W.columns:
        return pd.DataFrame()
    for v in [v for v in ORDER if v in W.columns and v != "none"]:
        fixed, broken = int((~W["none"] & W[v]).sum()), int((W["none"] & ~W[v]).sum())
        p = binomtest(fixed, fixed + broken).pvalue if fixed + broken else 1.0
        out.append(dict(version=v, accuracy=W[v].mean() * 100, change=(W[v].mean() - W["none"].mean()) * 100,
                        fixed=fixed, broken=broken, p=round(p, 4)))
    return pd.DataFrame(out).sort_values("change", ascending=False).set_index("version")

def factorial(t):
    cells = ["none", "brighten", "zoom", "brighten+zoom"]
    if not all(c in t.index for c in cells):
        return None
    n, b, z, bz = (t.loc[c] for c in cells)
    return pd.DataFrame({"L3 as taken": n, "brighten only": b, "zoom only": z, "both": bz,
                         "effect of brightening": (b + bz) / 2 - (n + z) / 2,
                         "effect of zoom": (z + bz) / 2 - (n + b) / 2,
                         "extra from doing both": (bz - z) - (b - n)}).T

TABLES = {}
for model in ["CLIP database", "CLIP zero-shot"]:
    TABLES[f"{model}: accuracy"] = acc_table(model)
    TABLES[f"{model}: vs as taken"] = versus_none(model)
    TABLES[f"{model}: brighten x zoom"] = factorial(TABLES[f"{model}: accuracy"])
    print(f"\n===== {model} =====")
    print("A. Accuracy (%) by noise type:")
    display(TABLES[f"{model}: accuracy"].round(1))
    print("B. Each version against the photo as taken (change in points, same photos):")
    display(TABLES[f"{model}: vs as taken"].round({"accuracy": 1, "change": 1, "p": 3}))
    if TABLES[f"{model}: brighten x zoom"] is not None:
        print("C. Brightening and zoom, separately and together (accuracy %, effects in points):")
        display(TABLES[f"{model}: brighten x zoom"].round(1))

# the best version for each noise type (used by the photo-check notebook to choose fixes)
t = TABLES["CLIP database: accuracy"]
best = []
for nz in NOISES:
    col = NOISE_NAME[nz]
    cand = t.loc[[v for v in VARIANTS if v in t.index], col]
    v = cand.idxmax() if cand.max() > cand.get("none", -1) else "none"
    best.append(dict(problem=nz, version=v, steps=json.dumps(VARIANTS[v]), accuracy=cand[v], as_taken=cand["none"],
                     gain=cand[v] - cand["none"]))
best.append(dict(problem="clean", version="none", steps="[]", accuracy=np.nan, as_taken=np.nan, gain=0.0))
BEST = pd.DataFrame(best)
BEST.to_csv(os.path.join(OUT_DIR, "best_fix_per_problem.csv"), index=False)
print("\nBest version for each noise type (CLIP database). Saved for the photo-check notebook:")
display(BEST.round(1))

## 7. Pipelines: L3 → L3′ → L3″

Each pipeline tries its versions **in order** and stops as soon as the CLIP database is confident (the winning disease has at least `STOP_CONFIDENCE` of the vote). If it is never confident, the last version's answer counts. Brightening and zoom are in **separate pipelines**, so you can see which one did the work.

* **Tries:** how many versions were looked at, on average. Unlike asking the farmer for another photo, a try here costs nothing but a second of computing.
* **Confident but wrong:** stopped with a confident answer that was wrong, the most harmful outcome.
* **Best possible:** at least one version in the pipeline was right. A better stop rule could get at most this.

In [ ]:
def run_pipeline(R, seq, stop):
    R = R.set_index(["image", "variant"])
    out = []
    for r in Q.itertuples(index=False):
        if (r.image, seq[0]) not in R.index:
            continue
        tries, last, ok_stop, any_right = 0, None, False, False
        for v in seq:
            x = R.loc[(r.image, v)]
            tries += 1
            any_right = any_right or bool(x.correct)
            last = x
            if stop(x):
                ok_stop = True
                break
        out.append(dict(image=r.image, noise=r.noise, tries=tries, correct=bool(last.correct), confident=ok_stop,
                        any_right=any_right))
    return pd.DataFrame(out)

def pipeline_table(R, stop):
    rows, by_noise = [], {}
    have = set(R.variant.unique())
    for name, seq in [("L3 as taken (1 try)", ["none"])] + list(PIPELINES.items()):
        if not set(seq) <= have:
            continue
        p = run_pipeline(R, seq, stop)
        if not len(p):
            continue
        rows.append({"pipeline": name, "accuracy (%)": p.correct.mean() * 100, "average tries": p.tries.mean(),
                     "stopped confident (%)": p.confident.mean() * 100,
                     "confident but wrong (%)": (p.confident & ~p.correct).mean() * 100,
                     "best possible (%)": p.any_right.mean() * 100})
        by_noise[name] = p.groupby("noise").correct.mean().mul(100).rename(index=NOISE_NAME)
    return (pd.DataFrame(rows).set_index("pipeline"),
            pd.DataFrame(by_noise).T.reindex(columns=[NOISE_NAME[n] for n in NOISES if NOISE_NAME[n] in pd.DataFrame(by_noise).index]))

PIPE, PIPE_NOISE = pipeline_table(RES[RES.model == "CLIP database"], lambda x: x.conf >= STOP_CONFIDENCE)
print("CLIP database pipelines:")
display(PIPE.round(1))
print("Accuracy (%) by noise type:")
display(PIPE_NOISE.round(1))

## 8. Gemini on the changed photos (optional, uses quota)

Gemini 3.5 Flash-Lite gets each version in `GEMINI_VARIANTS` with Step 2's prompt (the 33 option names). The photo as taken is not asked again: Step 2's answers are used. To save quota it uses `GEMINI_PHOTOS_PER_DISEASE` photos per disease, the same photos for every version, so the versions are compared on exactly the same images. Gemini's pipelines stop when it says the photo is clear enough with confidence of at least 0.8.

In [ ]:
GQ = gemini_queue(TEST_LEVELS, GEMINI_PHOTOS_PER_DISEASE)
GEM_FILES = {v: os.path.join(OUT_DIR, f"gemini_variant_{slug(v)}.csv") for v in VARIANTS.keys() | set(AUTO)}
for v in GEMINI_VARIANTS:
    status = run_job(f"Gemini: {v}", GEM_FILES[v], GQ,
                     lambda row, v=v: [plain_prompt(OPTIONS), part_from_array(variant_array(v, row))], OPTIONS)
    if status in ("daily", "key"):
        break

gq_images = {r.image for r in GQ}
parts = []
plain = load_answers(GEMINI_CACHE)
if len(plain):
    parts.append(plain[plain.image.isin(gq_images)].assign(variant="none"))
for v, path in GEM_FILES.items():
    a = load_answers(path)
    if len(a):
        parts.append(a[a.image.isin(gq_images)].assign(variant=v))
GRES = pd.DataFrame()
if parts:
    GRES = pd.concat(parts, ignore_index=True)
    GRES = GRES.rename(columns={"pred_label": "pred"}).merge(Q[["image", "photo_id", "disease", "group", "noise", "level", "detected"]], on="image")
    GRES["model"] = "Gemini"
    GRES["correct"] = GRES.pred == GRES.disease
    GRES["conf"] = pd.to_numeric(GRES.confidence, errors="coerce")
    complete = GRES.groupby("image").variant.nunique()
    GRES = GRES[GRES.image.isin(complete.index[complete == GRES.variant.nunique()])]   # same images for every version
if len(GRES) and GRES.variant.nunique() > 1:
    print(f"Gemini, {GRES.image.nunique()} images answered for every version:")
    TABLES["Gemini: accuracy"] = acc_table("Gemini", GRES)
    TABLES["Gemini: vs as taken"] = versus_none("Gemini", GRES)
    display(TABLES["Gemini: accuracy"].round(1))
    display(TABLES["Gemini: vs as taken"].round({"accuracy": 1, "change": 1, "p": 3}))
    TABLES["Gemini: brighten x zoom"] = factorial(TABLES["Gemini: accuracy"])
    if TABLES["Gemini: brighten x zoom"] is not None:
        display(TABLES["Gemini: brighten x zoom"].round(1))
    gstop = lambda x: str(x.clear_enough).lower() == "true" and (x.conf if pd.notna(x.conf) else 0) >= 0.8
    GPIPE, GPIPE_NOISE = pipeline_table(GRES, gstop)
    TABLES["Gemini: pipelines"] = GPIPE
    print("Gemini pipelines (only the versions Gemini was asked about):")
    display(GPIPE.round(1))
else:
    print("No Gemini answers for changed photos yet (GEMINI_VARIANTS is empty, or the quota ran out before the first one).")

## 9. Charts, tables and download

The heat map shows how many points each version **gained (blue)** or **lost (orange)** compared with the L3 photo as taken, for each noise type. Everything is saved in `SafeCrop/results/l3_variants`: `l3_variants_tables.xlsx`, the predictions for every version, `best_fix_per_problem.csv` (read by the photo-check notebook) and the figures.

In [ ]:
def change_heatmap(t, title, path):
    d = t.sub(t.loc["none"], axis=1).drop(index="none")
    lim = max(5.0, float(np.nanmax(np.abs(d.to_numpy()))))
    fig, ax = plt.subplots(figsize=(1.15 * d.shape[1] + 3.2, 0.36 * d.shape[0] + 1.4), dpi=150)
    im = ax.imshow(d.to_numpy(), cmap=matplotlib.colors.LinearSegmentedColormap.from_list(
        "gain", ["#eb6834", "#ffffff", "#2a78d6"]), vmin=-lim, vmax=lim, aspect="auto")
    for i in range(d.shape[0]):
        for j in range(d.shape[1]):
            v = d.iat[i, j]
            if pd.notna(v):
                ax.text(j, i, f"{v:+.0f}", ha="center", va="center", fontsize=8, color="#1f2937")
    ax.set_xticks(range(d.shape[1]), d.columns, fontsize=8.5, rotation=20, ha="right")
    ax.set_yticks(range(d.shape[0]), d.index, fontsize=8.5)
    ax.set_title(title, loc="left", fontsize=10.5)
    for s in ax.spines.values():
        s.set_visible(False)
    fig.colorbar(im, ax=ax, shrink=0.6, label="points vs as taken")
    fig.tight_layout()
    fig.savefig(path, facecolor="white")
    plt.close(fig)

change_heatmap(TABLES["CLIP database: accuracy"], f"CLIP database: change in accuracy vs the {LEVEL_SHOW} photo as taken (points)",
               os.path.join(FIG, "change_clip_database.png"))
change_heatmap(TABLES["CLIP zero-shot: accuracy"], f"CLIP zero-shot: change in accuracy vs the {LEVEL_SHOW} photo as taken (points)",
               os.path.join(FIG, "change_clip_zero_shot.png"))
if "Gemini: accuracy" in TABLES:
    change_heatmap(TABLES["Gemini: accuracy"], "Gemini: change in accuracy vs the photo as taken (points)",
                   os.path.join(FIG, "change_gemini.png"))

# brighten x zoom, CLIP database, per noise type
t = TABLES["CLIP database: accuracy"]
cells = [c for c in ["none", "brighten", "zoom", "brighten+zoom"] if c in t.index]
if len(cells) == 4:
    fig, ax = plt.subplots(figsize=(9, 3.8), dpi=150)
    x = np.arange(t.shape[1])
    for k, (c, colr) in enumerate(zip(cells, ["#9aa0a6", "#eb6834", "#2a78d6", "#1baf7a"])):
        ax.bar(x + (k - 1.5) * 0.2, t.loc[c], width=0.2, color=colr, label={"none": "as taken"}.get(c, c))
    ax.set_xticks(x, t.columns, fontsize=8.5)
    ax.set_ylabel("Accuracy (%)")
    ax.set_ylim(0, 100)
    ax.grid(axis="y", color="#e1e0d9", lw=0.7)
    ax.set_axisbelow(True)
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
    ax.legend(frameon=False, fontsize=8.5, ncol=4, loc="upper left")
    ax.set_title("CLIP database: brightening and zoom, separately and together", loc="left", fontsize=10.5)
    fig.tight_layout()
    fig.savefig(os.path.join(FIG, "brighten_and_zoom.png"), facecolor="white")
    plt.close(fig)

for f in ["change_clip_database.png", "brighten_and_zoom.png", "change_gemini.png", "change_clip_zero_shot.png"]:
    if os.path.exists(os.path.join(FIG, f)):
        display(ShowImage(filename=os.path.join(FIG, f), width=900))

# ---- the main findings in words ----
lines = []
vs = TABLES["CLIP database: vs as taken"]
helped = vs[(vs.change > 0) & (vs.p < 0.05)]
hurt = vs[(vs.change < 0) & (vs.p < 0.05)]
lines.append("CLIP database, versions that clearly helped overall: " +
             (", ".join(f"{v} ({r.change:+.0f})" for v, r in helped.iterrows()) or "none"))
lines.append("CLIP database, versions that clearly hurt overall: " +
             (", ".join(f"{v} ({r.change:+.0f})" for v, r in hurt.iterrows()) or "none"))
fz = TABLES["CLIP database: brighten x zoom"]
if fz is not None:
    lines.append(f"Brightening alone changed CLIP database accuracy by {fz.loc['effect of brightening', 'All']:+.0f} points, "
                 f"zoom by {fz.loc['effect of zoom', 'All']:+.0f}; low light only: brightening "
                 f"{fz.loc['effect of brightening', NOISE_NAME['low_light']]:+.0f}, zoom {fz.loc['effect of zoom', NOISE_NAME['low_light']]:+.0f}.")
for _, r in BEST.iterrows():
    if r.problem != "clean":
        lines.append(f"Best version for {NOISE_NAME[r.problem].lower()}: {r.version} ({r.accuracy:.0f}% vs {r.as_taken:.0f}% as taken).")
print("In short:")
for l in lines:
    print(" •", l)

xlsx = os.path.join(OUT_DIR, "l3_variants_tables.xlsx")
with pd.ExcelWriter(xlsx, engine="openpyxl") as w:
    pd.DataFrame({"summary": lines + [f"Levels tested: {', '.join(TEST_LEVELS)}", f"CLIP model: {CLIP_MODEL}",
                                      f"Stop confidence: {STOP_CONFIDENCE}"]}).to_excel(w, sheet_name="Summary", index=False)
    for name, tab in TABLES.items():
        if tab is not None and len(tab):
            tab.round(2).to_excel(w, sheet_name=re.sub(r"[^A-Za-z0-9 ]+", " ", name)[:31])
    PIPE.round(2).to_excel(w, sheet_name="CLIP database pipelines")
    PIPE_NOISE.round(2).to_excel(w, sheet_name="Pipelines by noise type")
    BEST.to_excel(w, sheet_name="Best fix per problem", index=False)
if len(GRES):
    GRES.to_csv(os.path.join(OUT_DIR, "gemini_predictions_by_version.csv"), index=False)
print("Saved", xlsx)
zip_path = shutil.make_archive(os.path.join(WORK, "SafeCrop_l3_variants_results"), "zip", OUT_DIR)
print("Zip ready:", zip_path, f"({os.path.getsize(zip_path) / 1e6:.1f} MB)")
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("Automatic download didn't start (" + short(e, 80) + "). Use the folder icon on the left to download it.")